# Content Popularity Tracker
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Hash Tables, Heaps · **Difficulty/Frequency:** Very Common (8/10)


## Concepts

**What this problem is really testing:**
- Bucketing items into groups by their current value
- Amortized complexity (why a loop that "looks slow" can still be cheap overall)
- A heap-based idea that seems reasonable but turns out to be flawed here

**Why each one shows up here:**
- This is a "keep track of the running maximum while values go up and down by 1" problem — the same shape as LeetCode's *All O`one Data Structure*.
- A plain hash map gives you O(1) updates, but then finding the max means scanning everything — O(n).
- A heap looks like the fix, but Python's `heapq` can't update an entry's priority in place — so you're back to scanning, just in a different disguise.
- The real fix uses a fact this problem gives you for free: **scores only ever change by exactly 1**. That means you can group ("bucket") content IDs by their current score, and just track which bucket is currently the highest.

**The one idea to hold onto:** don't find the max by searching for it. Keep the answer updated as a side effect of every change, and only do extra work in proportion to how far the answer actually needs to move.

---

### Quick primers — the building blocks used below

**What is a Hash Map?**
- A hash map (Python `dict`) stores key → value pairs by hashing the key to a slot — giving **O(1) average** insert/lookup/delete.
- **In Python:** use `dict` for key→value. Use `set` when you only need fast membership + removal — exactly what "the group of IDs currently at score s" needs.

**What is a Heap (priority queue)?**
- A binary heap keeps the min (or max) element accessible in O(1), with O(log n) push/pop, by keeping one rule true: "parent ≤ children".
- **The catch, specifically for this problem:** a normal heap has no fast (O(log n)) way to change one element's priority, or remove an item that isn't at the top.
- If a score can go down, your options are: rebuild the whole heap, or push a new entry and later throw away the old, now-stale one when it resurfaces (**lazy deletion**) — which can force `mostPopular()` to pop through several stale entries in a row.

**Amortized analysis (why a "slow-looking" loop can still be cheap).**
- A single operation can *look* expensive in the worst case — e.g. a `while` loop that walks through several values.
- But if you can show the **total** work across *every* call combined is bounded, then the *average* cost per call is small — even though no individual call is guaranteed to be fast.
- This is different from "best case" or "worst case" per call — it's a statement about a whole sequence of operations, not any one of them.

**Bucketing by value.**
- When values change in small, predictable steps (here, always ±1), you can group items into "buckets" keyed by their current value.
- Moving an item when its value changes becomes: remove from one bucket, add to another — both **O(1)** — instead of searching for anything.


## Problem Statement

Design `ContentPopularity` with:
- `increasePopularity(contentId)` -- +1 (thumbs up).
- `decreasePopularity(contentId)` -- -1 (thumbs down).
- `mostPopular()` -- the content ID with the highest popularity (any one, on ties); `-1` if nothing exists.

**Official follow-ups (covered below):** non-existent ID on decrease; top-K extension; scores that change by more than 1; removing content entirely; deterministic tie-breaking.


### Approach 1 -- Naive (hash map + linear scan)

**Idea:** a plain `dict` from content ID to score. `increase`/`decrease` are O(1) dict updates. `mostPopular()` scans every entry to find the max.

**Time complexity:** O(1) for `increase`/`decrease`; **O(n)** for `mostPopular()`, where n is the number of distinct IDs.

**Space complexity:** O(n).


In [1]:
from typing import Dict


class ContentPopularityNaive:
    def __init__(self) -> None:
        self.score: Dict[int, int] = {}

    def increasePopularity(self, contentId: int) -> None:
        self.score[contentId] = self.score.get(contentId, 0) + 1

    def decreasePopularity(self, contentId: int) -> None:
        if contentId in self.score:
            self.score[contentId] -= 1

    def mostPopular(self) -> int:
        if not self.score:
            return -1
        return max(self.score, key=self.score.get)   # O(n) scan every call

        # Equivalent explicit loop -- same O(n) scan, spelled out step by step:
        # best_id = None
        # best_score = None
        # for contentId in self.score:            # iterate over keys
        #     current = self.score[contentId]       # same as self.score.get(contentId)
        #     if best_score is None or current > best_score:
        #         best_id = contentId
        #         best_score = current
        # return best_id


### Step-by-step: what this cell actually does

All three methods share one piece of state, `self.score: Dict[int, int]` -- a plain hash map from content ID to its current popularity count. `increasePopularity` and `decreasePopularity` only ever touch their own ID's entry, so each is a single O(1) dict write. `mostPopular` has no shortcut: it walks every key in `self.score` to find the one with the largest value, so its cost grows with the number of distinct IDs ever tracked. The trace below runs the notebook's own demo sequence from the `ops` list in the Verification cell, which is exactly what proves `expected = [-1, 1, 2, 1, 2]`.

#### `increasePopularity(contentId)` -- +1, initializing at 0 if new

1. Read the current score for `contentId`, defaulting to `0` if the ID has never been seen (`self.score.get(contentId, 0)`).
2. Write back `current + 1`.

Demo: `increasePopularity(1)` when `self.score = {}`.

| Step | Action | `score` after |
|---|---|---|
| 1 | `self.score.get(1, 0)` -> `0` (not present, default used) | `{}` |
| 2 | write `self.score[1] = 0 + 1` | `{1: 1}` |

#### `decreasePopularity(contentId)` -- -1, no-op if unknown

1. Check whether `contentId` is a key in `self.score`.
2. If it isn't, do nothing and return (documented no-op for an unknown ID).
3. If it is, write back `current - 1` -- note there is no floor at `0`, so this can go negative.

Demo: `decreasePopularity(99)` when `self.score = {1: 1}` (99 was never tracked).

| Step | Action | `score` after |
|---|---|---|
| 1 | `99 in self.score` -> `False` | `{1: 1}` |
| 2 | guard fails, method returns immediately -- no write | `{1: 1}` |

#### `mostPopular()` -- linear scan for the max

1. If `self.score` is empty, return `-1` (nothing tracked yet).
2. Otherwise call `max(self.score, key=self.score.get)`, which walks **every** key, looks up its value, and keeps the key with the largest value seen so far.
3. Return that key. Python's `max` keeps the *first* maximal item it encounters, so among tied scores the earliest-inserted key wins -- the notebook does not rely on any particular tie-break.

Demo: the full `ops` sequence from the Verification cell, run against a fresh `ContentPopularityNaive()`.

| Step | Action | `score` after | `mostPopular()` returns |
|---|---|---|---|
| 1 | `mostPopular()` on empty map | `{}` | `-1` |
| 2 | `increasePopularity(1)` | `{1: 1}` | -- |
| 3 | `increasePopularity(1)` | `{1: 2}` | -- |
| 4 | `increasePopularity(2)` | `{1: 2, 2: 1}` | -- |
| 5 | `mostPopular()`: scan `{1: 2, 2: 1}`, max value 2 -> key `1` | `{1: 2, 2: 1}` | `1` |
| 6 | `increasePopularity(2)` | `{1: 2, 2: 2}` | -- |
| 7 | `increasePopularity(2)` | `{1: 2, 2: 3}` | -- |
| 8 | `mostPopular()`: scan `{1: 2, 2: 3}`, max value 3 -> key `2` | `{1: 2, 2: 3}` | `2` |
| 9 | `decreasePopularity(2)` | `{1: 2, 2: 2}` | -- |
| 10 | `decreasePopularity(2)` | `{1: 2, 2: 1}` | -- |
| 11 | `mostPopular()`: scan `{1: 2, 2: 1}`, max value 2 -> key `1` | `{1: 2, 2: 1}` | `1` |
| 12 | `decreasePopularity(1)` | `{1: 1, 2: 1}` | -- |
| 13 | `decreasePopularity(1)` | `{1: 0, 2: 1}` | -- |
| 14 | `mostPopular()`: scan `{1: 0, 2: 1}`, max value 1 -> key `2` | `{1: 0, 2: 1}` | `2` |

Final returns across the five `mostPopular()` calls are `[-1, 1, 2, 1, 2]` -- matches `expected = [-1, 1, 2, 1, 2]` in the Verification cell. Row 14 is the interesting one: ID `1` was the max as recently as row 11, but two decreases (rows 12-13) drop it below ID `2` without either ID's bucket ever being examined -- `mostPopular()` only finds this out because it rescans *everything* on the next call.

#### Mental model

- All three methods read/write the *same* `self.score` dict -- there's no auxiliary structure, which is exactly why `increase`/`decrease` are trivially O(1) but `mostPopular` has nothing faster than a full scan to fall back on.
- The cost is invisible per-call but adds up: call `mostPopular()` after every `increasePopularity()` for `n` distinct IDs and the total work is O(1 + 2 + ... + n) = O(n^2), which is what the Empirical complexity check cell further down measures directly.
- "Initialize at 0 on first increase" (`.get(contentId, 0)`) and "no-op on decrease of an unknown ID" (`if contentId in self.score`) are two independent policy choices baked into this naive version -- Approaches 2 and 3 keep the same two choices, so the three classes stay behaviorally interchangeable and comparable in the Verification cell.
- This class is the baseline the rest of the notebook measures against: Approach 2 fixes nothing about the scan (it just moves the cost into `mostPopular`'s stale-entry cleanup), and only Approach 3's bucket-by-value trick actually removes the linear scan.


### Approach 2 -- Heap with lazy deletion (why it *almost* works)

**Idea:** push `(-score, contentId)` onto a max-heap-via-negation whenever a score changes. `mostPopular()` peeks the top, but the top might be **stale** (that ID's score has since changed via a newer push) -- so keep popping until the top entry's stored score matches the ID's *current* score in a side `dict`.

**Time complexity:** each push is O(log n). `mostPopular()` is O(log n) **amortized** across all calls in the best case, but a single call can pop through an arbitrary run of stale entries first -- there's no per-call guarantee, only an amortized one tied to total pushes.

**Space complexity:** O(n + P) where P is the total number of pushes ever made -- stale entries accumulate in the heap until they happen to surface and get discarded.


In [2]:
import heapq
from typing import Dict, List, Tuple


class ContentPopularityHeap:
    def __init__(self) -> None:
        self.score: Dict[int, int] = {}
        self.heap: List[Tuple[int, int]] = []   # (-score, contentId), possibly stale

    def _push(self, contentId: int) -> None:
        heapq.heappush(self.heap, (-self.score[contentId], contentId))

    def increasePopularity(self, contentId: int) -> None:
        self.score[contentId] = self.score.get(contentId, 0) + 1
        self._push(contentId)

    def decreasePopularity(self, contentId: int) -> None:
        if contentId not in self.score:
            return
        self.score[contentId] -= 1
        self._push(contentId)

    def mostPopular(self) -> int:
        while self.heap:
            neg_score, contentId = self.heap[0]
            if -neg_score == self.score.get(contentId):   # still current -- not stale
                return contentId
            heapq.heappop(self.heap)                       # stale entry, discard and keep looking
        return -1


### The easy way to think about it

- **The problem with a normal heap:** a heap is great at "give me the biggest thing," but terrible at "this specific thing's value just changed -- please re-sort it." Python's `heapq` has no built-in way to update a priority in place.

**The workaround -- "push new, ignore old":**

- Instead of updating an entry, `_push` just pushes a brand new one every time a score changes, and leaves the old one sitting in the heap as junk:
  ```python
  def _push(self, contentId):
      heapq.heappush(self.heap, (-self.score[contentId], contentId))
  ```
- It's negated because `heapq` is a *min*-heap, so pushing `-score` makes the smallest number = the largest score, which pops out first.

**Analogy:** imagine a leaderboard where you never erase old scores -- every time someone's score changes, you tape a *new* sticky note with their new score on top of the pile. The old sticky note for that person is still stuck in there somewhere, just outdated.

**So how do you get the right answer?** When someone asks "who's winning right now," look at the topmost sticky note. But that note might be old -- maybe that person's score has changed since then. Check: does this note's score match what's currently written down for them in `self.score`?

- If yes -> trust it, that's the answer.
- If no -> it's a stale note, throw it away, and check the next one underneath.

```python
def mostPopular(self):
    while self.heap:
        neg_score, contentId = self.heap[0]         # peek the top note
        if -neg_score == self.score.get(contentId):  # is it still true?
            return contentId                          # yes -> done
        heapq.heappop(self.heap)                      # no -> stale, throw it away, keep checking
    return -1
```

**Tiny concrete example:**

```
increasePopularity(1)   # score={1:1}, note pile top: "1 has score 1"
increasePopularity(1)   # score={1:2}, new note on top: "1 has score 2"
decreasePopularity(1)   # score={1:1}, new note on top: "1 has score 1"
                         #   -- now 2 outdated notes are buried: "1 has score 2" and the earlier "1 has score 1"
mostPopular()
  -> peek top note: could be "1 has score 2" (the highest note ever pushed)
  -> check: self.score[1] is actually 1, not 2 -> STALE, throw it away
  -> peek next: "1 has score 1" -> matches self.score[1]=1 -> real, return 1
```

**Why this "almost works":** it's correct, but you pay for it in cleanup -- every stale note has to be thrown away one by one before finding a true one. If someone's score bounces up and down a lot, a bunch of garbage notes pile up and a single `mostPopular()` call might dig through many of them before finding the real answer. That's why it's only *amortized* fast, not guaranteed fast per call -- unlike the bucket approach (Approach 3), which never leaves stale junk behind at all.


### Step-by-step: what this cell actually does

- Shared state: `self.score` (current value per ID) and `self.heap` (a min-heap of `(-score, contentId)`, used as a max-heap via negation).
- Every `increasePopularity`/`decreasePopularity` **pushes a fresh entry** -- it never removes or updates the old one, so old entries can go stale.
- `mostPopular()` peeks the top; if the top's stored score no longer matches `self.score[contentId]`, that entry is stale -- pop and discard it, repeat.

#### `increasePopularity` / `decreasePopularity` -- push, never remove

- Update `self.score[contentId]` by ±1 (decrease is a no-op if the ID is unknown).
- Call `_push(contentId)`, which does `heapq.heappush(self.heap, (-self.score[contentId], contentId))`.
- Old entries for the same ID are left behind in the heap -- they're now potentially stale.

#### `mostPopular()` -- peek, discard stale, repeat

- Loop while the heap isn't empty:
  - Peek `heap[0]` = `(neg_score, contentId)`.
  - If `-neg_score == self.score.get(contentId)` -> still current, return `contentId`.
  - Otherwise -> stale, `heappop()` it and keep looping.
- Return `-1` if the heap empties out without finding a current entry.

**Demo** (small, deliberate sequence to force one stale pop):

```python
cp = ContentPopularityHeap()
cp.increasePopularity(1)   # score={1:1}, heap=[(-1,1)]
cp.increasePopularity(1)   # score={1:2}, push(-2,1) -> heap top = (-2,1)
cp.decreasePopularity(1)   # score={1:1}, push(-1,1) -- top (-2,1) is now STALE
cp.mostPopular()           # -> 1
```

| Step | Action | `score` | heap top peeked |
|---|---|---|---|
| 1 | `increasePopularity(1)` | `{1: 1}` | -- |
| 2 | `increasePopularity(1)` | `{1: 2}` | -- |
| 3 | `decreasePopularity(1)` | `{1: 1}` | -- |
| 4 | `mostPopular()`: peek `(-2, 1)` -> `2 != score[1]=1` -> **STALE**, pop it | `{1: 1}` | `(-2, 1)` discarded |
| 5 | `mostPopular()`: peek `(-1, 1)` -> `1 == score[1]=1` -> match, **return `1`** | `{1: 1}` | `(-1, 1)` |

This same class also passes the notebook's full `ops` sequence in Verification (`expected = [-1, 1, 2, 1, 2]`) -- the stale-pop above is the mechanism that makes that agreement possible even as scores go up and down.

#### Mental model

- Pushing instead of updating trades an O(log n) "decrease-key" (which `heapq` can't do) for O(log n) pushes plus **occasional** cleanup in `mostPopular()`.
- A single `mostPopular()` call has **no fixed worst-case bound** -- it can pop through an arbitrary run of stale entries; the guarantee is only amortized across all pushes ever made (space grows unboundedly with churn, unlike the bucket approach).
- This is why "push a new entry, lazily discard the old one" is a common heap workaround, but not a free one -- it shows up again as a reusable pattern (e.g. Dijkstra with decrease-key simulated the same way).


### Approach 3 -- Optimal (score buckets + a max pointer)

**Idea:** maintain `score[id] -> current popularity`, `buckets[s] -> set of ids at score s`, and a `max_score` pointer. `increase`/`decrease` move an ID between two adjacent buckets -- pure set add/discard, O(1). If the *old* max bucket becomes empty on a decrease, walk `max_score` down until a non-empty bucket is found.

**Time complexity:** O(1) for `increase` and `mostPopular()`. Because `_move` deletes a bucket the instant it empties, `buckets` never contains gaps -- so whenever `decreasePopularity` needs to walk `max_score` down, `new_score` (exactly one below the old max) is *already* guaranteed non-empty (it just received the ID that moved). The walk-down is therefore **exactly one step, every time**, not merely "amortized O(1)" -- worst case O(1) too.

**Space complexity:** O(n) -- `score` holds one entry per ID, and every ID lives in exactly one bucket's set at a time.

> **A bug worth catching.** The source material's own reference answer bounds that walk-down loop with `while self.max_score > 0 and ...`. That extra `> 0` silently assumes scores never go negative -- but nothing in the problem statement guarantees that (a content ID can rack up more thumbs-down than thumbs-up). Once a lone ID's score crosses from 0 to -1, that guard stops the walk at `max_score = 0`, `mostPopular()` then finds bucket 0 empty and returns **-1** even though the ID is still being tracked. Since the walk only ever needs to move exactly one step (see above), the fix is simply to drop the artificial floor -- shown in the code below and caught by the "score goes negative" edge case in Verification.


In [3]:
from typing import Dict, Set


class ContentPopularity:
    def __init__(self) -> None:
        self.score: Dict[int, int] = {}
        self.buckets: Dict[int, Set[int]] = {}
        self.max_score = 0

    def _move(self, contentId: int, old_score: int, new_score: int) -> None:
        if old_score in self.buckets:
            self.buckets[old_score].discard(contentId)
            if not self.buckets[old_score]:
                del self.buckets[old_score]           # keep buckets sparse -- only non-empty scores
        self.buckets.setdefault(new_score, set()).add(contentId)
        self.score[contentId] = new_score

    def increasePopularity(self, contentId: int) -> None:
        old_score = self.score.get(contentId, 0)
        new_score = old_score + 1
        self._move(contentId, old_score, new_score)
        if new_score > self.max_score:
            self.max_score = new_score

    def decreasePopularity(self, contentId: int) -> None:
        if contentId not in self.score:
            return
        old_score = self.score[contentId]
        new_score = old_score - 1
        self._move(contentId, old_score, new_score)
        if old_score == self.max_score and old_score not in self.buckets:
            # No floor at 0: scores CAN go negative (see the note below), and `new_score`
            # (old_score - 1) is always guaranteed non-empty -- it just received `contentId` --
            # so this loop is guaranteed to terminate there even without an artificial floor.
            while self.max_score not in self.buckets:
                self.max_score -= 1

    def mostPopular(self) -> int:
        if not self.buckets or self.max_score not in self.buckets:
            return -1
        return next(iter(self.buckets[self.max_score]))   # any ID in the max bucket, O(1)


### The easy way to think about it

- **The problem with the naive dict and the heap:** both need to *search* for the max -- the dict scans everything, the heap has to dig past stale junk. Neither one *knows* the answer instantly.

**The trick: sort people into labeled folders, and remember which folder is currently on top.**

- Instead of one big pile of scores, keep a folder for every score value: `buckets[5]` holds the set of every ID currently at score 5, `buckets[3]` holds every ID at score 3, and so on.
- Also keep one sticky note, `max_score`, saying "the highest folder that currently has anyone in it is folder number ___."
- When someone's score changes, you don't search anything -- you just physically move their ID from their old folder to the new one (`_move`):
  ```python
  self.buckets[old_score].discard(contentId)   # take them out of the old folder
  self.buckets[new_score].add(contentId)        # put them in the new folder
  ```

**`increasePopularity`:** move the ID up one folder. If that new folder number is higher than the `max_score` note, update the note -- done, O(1).

**`decreasePopularity`:** move the ID down one folder. The only tricky part: what if that was the *last* person in the top folder, and now it's empty? Then the `max_score` note is lying -- you have to peek one folder down. But here's the shortcut: the person who just moved *landed exactly one folder below* -- so that folder is *guaranteed* to have someone in it already. You never have to peek more than once.

```python
if old_score == self.max_score and old_score not in self.buckets:
    while self.max_score not in self.buckets:
        self.max_score -= 1        # only ever runs once
```

**`mostPopular()`:** just open the folder named on the sticky note and grab anyone inside -- O(1), no searching at all:
```python
return next(iter(self.buckets[self.max_score]))
```

**Tiny concrete example:**

```
increasePopularity(1)   # folder 1: {1}          max_score note: 1
increasePopularity(2)   # folder 1: {1, 2}        max_score note: 1
increasePopularity(2)   # folder 1: {1}, folder 2: {2}   max_score note: 2
decreasePopularity(2)   # 2 moves from folder 2 -> folder 1
                         #   folder 2 is now EMPTY, and it was the max folder
                         #   -> peek one folder down: folder 1 -> has {1, 2} -> update note to 1
mostPopular()            # open folder 1 -> return 1 or 2
```

**Why this beats the dict and the heap:** nothing is ever searched or left behind as junk. Every ID lives in exactly one folder at all times, and the sticky note is always kept truthful the instant a folder empties -- so both updates and "who's winning" are pure O(1), no exceptions.


### Step-by-step: what this cell actually does

- Shared state: `self.score` (current value per ID), `self.buckets` (`dict[score] -> set[ids]`, only non-empty scores present), `self.max_score` (pointer to the currently-highest occupied bucket).
- `_move` is the one primitive both public methods use: remove the ID from its old bucket (deleting the bucket if it goes empty), add it to the new bucket, update `self.score`.
- `mostPopular()` never scans -- it just reads `self.buckets[self.max_score]`.

#### `increasePopularity(contentId)` -- move up one bucket

- `old_score = self.score.get(contentId, 0)`, `new_score = old_score + 1`.
- `_move(contentId, old_score, new_score)`.
- If `new_score > self.max_score`, bump the pointer up.

#### `decreasePopularity(contentId)` -- move down one bucket, then re-anchor the pointer

- No-op if `contentId` isn't tracked.
- `old_score = self.score[contentId]`, `new_score = old_score - 1`; `_move(contentId, old_score, new_score)`.
- If the *old* bucket was the max bucket and it's now gone (emptied), walk `self.max_score` down until it lands on a non-empty bucket. Because `_move` just populated `new_score = max_score - 1`, that walk is **always exactly one step**.

**Demo** (small sequence hitting the walk-down case):

```python
cp = ContentPopularity()
cp.increasePopularity(1)   # buckets={1:{1}},        max_score=1
cp.increasePopularity(2)   # buckets={1:{1,2}},       max_score=1
cp.increasePopularity(2)   # buckets={1:{1}, 2:{2}},  max_score=2
cp.decreasePopularity(2)   # 2 moves 2->1; bucket 2 empties -> walk max_score down
cp.mostPopular()           # -> 1 or 2 (both at score 1)
```

| Step | Action | `buckets` after | `max_score` after |
|---|---|---|---|
| 1 | `increasePopularity(1)`: `_move(1, 0, 1)`, `1 > max_score(0)` -> bump | `{1: {1}}` | `1` |
| 2 | `increasePopularity(2)`: `_move(2, 0, 1)`, `1 > max_score(1)`? no | `{1: {1, 2}}` | `1` |
| 3 | `increasePopularity(2)`: `_move(2, 1, 2)`, `2 > max_score(1)` -> bump | `{1: {1}, 2: {2}}` | `2` |
| 4 | `decreasePopularity(2)`: `_move(2, 2, 1)` -- bucket `2` empties and is deleted; `2` added to bucket `1` | `{1: {1, 2}}` | -- |
| 4b | `old_score(2) == max_score(2)` and `2 not in buckets` -> walk: `max_score -= 1` -> `1`, `1 in buckets` -> stop (one step) | `{1: {1, 2}}` | `1` |
| 5 | `mostPopular()`: `buckets[1] = {1, 2}` -> return any member | `{1: {1, 2}}` | `1` |

This walk-once guarantee is exactly what the Verification cell's full `ops` sequence and the O(n) benchmark in "Empirical complexity check" both rely on.

#### Mental model

- Bucketing turns "find the max" into "read a pointer" -- the work moves from `mostPopular()` (free) into keeping `max_score` honest on every decrease.
- The walk-down is **worst-case O(1)**, not just amortized, *because* `_move` guarantees the bucket one below the old max is already non-empty (it just received the moving ID) -- this only holds because scores change by exactly ±1.
- The buggy reference implementation's `while self.max_score > 0` guard breaks silently once a score goes negative; dropping the artificial floor (as this cell does) is what the "score goes negative" edge case in Verification catches.


## Verification

Run a scripted sequence against all three implementations and confirm they agree, plus check the edge cases each Talking Point calls out.

In [ ]:
def run_ops(cp, ops):
    """ops: list of ('inc'|'dec'|'top', contentId_or_None) -> list of mostPopular() results."""
    results = []
    for kind, arg in ops:
        if kind == "inc":
            cp.increasePopularity(arg)
        elif kind == "dec":
            cp.decreasePopularity(arg)
        else:
            results.append(cp.mostPopular())
    return results


ops = [
    ("top", None),          # -1, nothing exists yet
    ("inc", 1), ("inc", 1), ("inc", 2),
    ("top", None),          # 1 has score 2, the max
    ("inc", 2), ("inc", 2),
    ("top", None),          # 2 now has score 3, the max
    ("dec", 2), ("dec", 2),
    ("top", None),          # 2 back to 1, 1 has 2 -> max is 1
    ("dec", 1), ("dec", 1),
    ("top", None),          # 1 at 0, 2 at 1 -> max is 2
]
expected = [-1, 1, 2, 1, 2]

for cls in (ContentPopularityNaive, ContentPopularityHeap, ContentPopularity):
    got = run_ops(cls(), ops)
    assert got == expected, f"{cls.__name__} mismatch: {got} != {expected}"

# Edge cases
cp = ContentPopularity()
assert cp.mostPopular() == -1                    # nothing tracked yet
cp.decreasePopularity(99)                        # decrease on unknown id: documented no-op
assert cp.mostPopular() == -1
cp.increasePopularity(5)
assert cp.mostPopular() == 5
cp.decreasePopularity(5)
cp.decreasePopularity(5)                          # score goes negative; still the only id tracked
assert cp.mostPopular() == 5

# Tie handling: whichever id is returned must actually be at the max score
cp2 = ContentPopularity()
cp2.increasePopularity(10)
cp2.increasePopularity(20)
assert cp2.mostPopular() in (10, 20)
assert cp2.score[cp2.mostPopular()] == max(cp2.score.values())

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Top-K extension.** The bucket approach generalizes: to answer "top K IDs", you can't stop at just `max_score` -- walk buckets downward from `max_score`, collecting IDs until you have K (still efficient if K is small and scores are clustered near the top), or maintain a sorted structure over non-empty bucket keys (e.g. a small heap of bucket scores) for faster descent.
- **Scores that jump by more than 1 (a "super-like" worth +10).** The bucket structure itself still works -- `_move` doesn't care about the size of the jump. What breaks is the *amortized argument* for `decreasePopularity`'s walk-down: it relied on scores changing by exactly 1, so `max_score` only ever needs to step down one bucket at a time to find the next non-empty one. With arbitrary jumps, a big score could vacate a bucket far below any populated one, and you'd need a different structure (e.g. a sorted container of occupied scores, such as a heap of bucket keys or a balanced BST) to find the next-highest occupied score in better than O(range of scores).
- **Removing content entirely.** Add a `removeContent(contentId)` that deletes it from `score` and its current bucket, then re-runs the same "walk `max_score` down if its bucket is now empty" check as `decreasePopularity` -- shown below.
- **Deterministic tie-breaking (e.g. always the smallest ID).** Swap each bucket's `set` for a structure with a fast "give me the smallest" operation -- a small heap, or a sorted container -- which changes bucket operations from O(1) to O(log bucket size).


In [4]:
class ContentPopularityWithRemoval(ContentPopularity):
    """Bonus: supports fully removing a content id from tracking."""

    def removeContent(self, contentId: int) -> None:
        if contentId not in self.score:
            return
        old_score = self.score.pop(contentId)
        self.buckets[old_score].discard(contentId)
        if not self.buckets[old_score]:
            del self.buckets[old_score]
        if old_score == self.max_score and old_score not in self.buckets:
            # Unlike decreasePopularity, removal doesn't guarantee a non-empty bucket just
            # below -- the removed id isn't moved anywhere -- so this walk can run all the
            # way out (everything removed). Guard on `self.buckets`, not an arbitrary floor.
            while self.buckets and self.max_score not in self.buckets:
                self.max_score -= 1
            if not self.buckets:
                self.max_score = 0


cp = ContentPopularityWithRemoval()
cp.increasePopularity(1)
cp.increasePopularity(2)
cp.increasePopularity(2)
assert cp.mostPopular() == 2
cp.removeContent(2)
assert cp.mostPopular() == 1
cp.removeContent(1)
assert cp.mostPopular() == -1          # everything removed
print("removeContent works as expected.")


removeContent works as expected.


## Empirical complexity check

The interesting comparison is Approach 1 (naive scan) vs. Approach 3 (buckets) under a workload that calls `mostPopular()` after *every* `increasePopularity()` -- the worst case for the naive scan. If the naive version is really O(n) per `mostPopular()` call, running n such pairs costs O(n^2) total; the bucket version should stay O(n) total.

| Growth when n doubles | Implies |
|---|---|
| ~2x | linear (bucket approach) |
| ~4x | quadratic (naive approach) |


In [5]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def run_naive_workload(ids):
    cp = ContentPopularityNaive()
    for cid in ids:
        cp.increasePopularity(cid)
        cp.mostPopular()          # O(n) scan, called n times -> O(n^2) total


def run_bucket_workload(ids):
    cp = ContentPopularity()
    for cid in ids:
        cp.increasePopularity(cid)
        cp.mostPopular()          # O(1), called n times -> O(n) total


def make_worst_case(n):
    return (list(range(n)),)      # n distinct ids, each incremented once


solutions = {
    "naive scan (O(n) per call)": run_naive_workload,
    "buckets (O(1) per call)": run_bucket_workload,
}
sizes = [500, 1000, 2000, 4000]   # kept small: the naive side is quadratic
benchmark(solutions, make_worst_case, sizes, plot=True)



naive scan (O(n) per call)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
      500 |       3.90 |           n/a
     1000 |      13.95 |         3.57x
     2000 |      54.37 |         3.90x
     4000 |     206.27 |         3.79x

buckets (O(1) per call)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
      500 |       0.38 |           n/a
     1000 |       0.55 |         1.44x
     2000 |       1.03 |         1.88x
     4000 |       2.31 |         2.25x

Ratios per doubling: ~1x=>constant/log, ~2x=>linear or n log n, ~4x=>quadratic, ~8x=>cubic.

[plot skipped] matplotlib not installed - run: pip install matplotlib


{'naive scan (O(n) per call)': [3.903600008925423,
  13.949799991678447,
  54.37129999336321,
  206.2685999990208],
 'buckets (O(1) per call)': [0.3804000007221475,
  0.5478999955812469,
  1.029900013236329,
  2.3136999952839687]}

## 🧩 Patterns Learned

- **Bucket by value when updates are small, predictable steps.** Grouping items by their current value into a `dict[value] -> set[item]` turns "this item's value changed" into O(1) set moves -- the same idea behind bucket sort and counting sort.
- **Track a running extreme instead of recomputing it.** Maintaining `max_score` incrementally and only "walking it down" when its own bucket empties avoids ever scanning the full collection.
- **Amortized analysis: bound total work, not per-call work.** The walk-down loop in `decreasePopularity` looks like it could be O(n), but each score value is vacated at most once across the entire sequence of calls -- so total work is bounded, even though no single call has a tight worst-case bound on its own.
- **A heap isn't automatically the answer for "track the max under updates."** Heaps excel at "give me the extreme, then remove it" -- they don't support efficient arbitrary-key updates, which forces lazy deletion and reintroduces the scan you were trying to avoid.
- **State your policy for "operate on an ID that doesn't exist yet" explicitly.** No-op, initialize at zero, or raise -- any is defensible, but silently picking one without saying so is where interview points are lost.
- **Related problems:** LeetCode "All O`one Data Structure" (the direct analogue), LFU cache (bucket-by-frequency is the same trick), sliding window maximum (different technique, same "avoid recomputation" spirit).
- **Common pitfalls:** forgetting to delete empty bucket entries (leaks memory and breaks "is this score occupied" checks); assuming a heap gives O(log n) decrease-key for free; not deciding what "most popular" means when everything is tied or nothing has been added yet.
